In [ ]:

import streamlit as st
import ollama
import chromadb
import uuid
from pypdf import PdfReader

st.set_page_config(page_title="My RAG Chatbot")
st.title("📚 Chat with Your PDF")

LLM = "tinyllama"
EMBED = "all-minilm"

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection("documents")


def split_text(text, size=500, overlap=80):
    chunks = []
    start = 0

    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap

    return [c for c in chunks if c.strip()]


def process_pdf(file):
    reader = PdfReader(file)
    chunks = []

    for page_num, page in enumerate(reader.pages, 1):
        text = page.extract_text() or ""

        for chunk in split_text(text):
            chunks.append({
                "text": chunk,
                "page": page_num,
                "source": file.name
            })

    return chunks


def ingest(file):
    chunks = process_pdf(file)

    if not chunks:
        return 0

    texts = [c["text"] for c in chunks]

    embeddings = ollama.embed(
        model=EMBED,
        input=texts
    )["embeddings"]

    # Remove old version of this PDF
    collection.delete(where={"source": file.name})

    collection.add(
        ids=[str(uuid.uuid4()) for _ in chunks],
        documents=texts,
        embeddings=embeddings,
        metadatas=[
            {"source": c["source"], "page": c["page"]}
            for c in chunks
        ]
    )

    return len(chunks)


def ask_question(question):
    embedding = ollama.embed(
        model=EMBED,
        input=question
    )["embeddings"][0]

    results = collection.query(
        query_embeddings=[embedding],
        n_results=min(3, collection.count())
    )

    context = "\n\n".join(results["documents"][0])
    metadata = results["metadatas"][0]

    prompt = f"""
    Answer using only the context below.
    If the answer is missing, say you don't know.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    response = ollama.chat(
        model=LLM,
        messages=[{"role": "user", "content": prompt}],
        options={"num_ctx": 1024, "num_predict": 150}
    )

    sources = list({
        f"{m['source']} - Page {m['page']}"
        for m in metadata
    })

    return response["message"]["content"], sources


uploaded = st.file_uploader("Upload PDF", type="pdf")

if uploaded and st.button("Process PDF"):
    with st.spinner("Processing document..."):
        try:
            count = ingest(uploaded)
            if count:
                st.success(f"Indexed {count} chunks!")
            else:
                st.warning("No readable text found.")
        except Exception as e:
            st.error(str(e))

st.write("Total indexed chunks:", collection.count())

question = st.chat_input("Ask a question about your PDF")

if question:
    st.chat_message("user").write(question)

    if collection.count() == 0:
        st.warning("Upload and process a PDF first.")
    else:
        with st.chat_message("assistant"):
            try:
                with st.spinner("Finding answers..."):
                    answer, sources = ask_question(question)

                st.write(answer)
                st.write("Sources:", ", ".join(sources))

            except Exception as e:
                st.error(str(e))

run

In [ ]:
cd rag-project

In [ ]:
import os

os.makedirs("/content/rag-project", exist_ok=True)
os.chdir("/content/rag-project")

print("Project folder created!")
print(os.getcwd())

In [ ]:
!pip install -q streamlit chromadb pypdf ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!apt-get update -qq
!apt-get install -y zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama server started!")


In [ ]:
!ollama pull tinyllama


In [ ]:
!ollama pull all-minilm

In [ ]:
!ollama list


In [ ]:
import ollama

response = ollama.chat(
    model="tinyllama",
    messages=[
        {
            "role": "user",
            "content": "Explain RAG in two sentences."
        }
    ]
)

print(response["message"]["content"])

In [ ]:
embedding = ollama.embed(
    model="all-minilm",
    input="Retrieval Augmented Generation"
)

print("Embedding dimensions:", len(embedding["embeddings"][0]))

In [ ]:
from google.colab import files
from pypdf import PdfReader
import io

uploaded = files.upload()

filename = next(iter(uploaded))

reader = PdfReader(io.BytesIO(uploaded[filename]))

text = ""

for page_num, page in enumerate(reader.pages, 1):
    text += f"\n[Page {page_num}]\n"
    text += page.extract_text() or ""

print("PDF:", filename)
print("Extracted characters:", len(text))
print(text[:1000])

In [ ]:
!pip install -q beautifulsoup4


In [ ]:
from google.colab import files
from pypdf import PdfReader
from bs4 import BeautifulSoup
import io
import os

uploaded = files.upload()

filename = next(iter(uploaded))
file_content = uploaded[filename]

text = ""

if filename.lower().endswith(".pdf"):

    reader = PdfReader(io.BytesIO(file_content))

    for page_num, page in enumerate(reader.pages, 1):
        text += f"\n[Page {page_num}]\n"
        text += page.extract_text() or ""

elif filename.lower().endswith((".html", ".htm")):

    soup = BeautifulSoup(file_content, "html.parser")

    for element in soup(["script", "style", "noscript"]):
        element.decompose()

    text = soup.get_text(separator="\n", strip=True)

else:
    raise ValueError("Unsupported file format!")

print("File:", filename)
print("Extracted characters:", len(text))
print("\nPreview:")
print(text[:1500])

In [ ]:
chunks = split_text(text)

print("Total chunks:", len(chunks))
print("First chunk:")
print(chunks[0])

In [ ]:
embeddings = ollama.embed(
    model="all-minilm",
    input=chunks
)["embeddings"]

print("Embedding generation successful!")
print("Embedding dimensions:", len(embeddings[0]))

In [ ]:
import chromadb
import uuid

client = chromadb.PersistentClient(
    path="/content/chroma_db"
)

collection = client.get_or_create_collection(
    name="business_ideas"
)

# Store document chunks and embeddings
collection.add(
    ids=[str(uuid.uuid4()) for _ in chunks],
    documents=chunks,
    embeddings=embeddings,
    metadatas=[
        {"source": filename, "chunk": i}
        for i in range(len(chunks))
    ]
)

print("Successfully stored embeddings!")
print("Total vectors:", collection.count())

In [ ]:
question = "What are some profitable business ideas?"

query_embedding = ollama.embed(
    model="all-minilm",
    input=question
)["embeddings"][0]

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

for i, document in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}:")
    print(document[:500])

In [ ]:
context = "\n\n".join(results["documents"][0])

prompt = f"""
You are a business consultant.

Answer the question using only the
following document context.

Context:
{context}

Question:
{question}

Provide a clear and concise answer.
"""

response = ollama.chat(
    model="tinyllama",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    options={
        "num_ctx": 1024,
        "num_predict": 200
    }
)

print("QUESTION:", question)
print("\nAI ANSWER:")
print(response["message"]["content"])

In [ ]:
!pip install -q streamlit

In [ ]:
%%writefile /content/app.py

import streamlit as st
import ollama
import chromadb
import uuid
import io

from bs4 import BeautifulSoup
from pypdf import PdfReader

st.set_page_config(
    page_title="RAG Business Assistant",
    page_icon="📚"
)

st.title("📚 RAG Business Assistant")
st.caption("Powered by Ollama, ChromaDB and Python")

client = chromadb.PersistentClient(
    path="/content/chroma_db"
)

collection = client.get_or_create_collection(
    name="business_ideas"
)


def extract_text(file):

    name = file.name.lower()

    if name.endswith(".pdf"):

        reader = PdfReader(io.BytesIO(file.getvalue()))

        return "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    elif name.endswith((".html", ".htm")):

        soup = BeautifulSoup(
            file.getvalue(),
            "html.parser"
        )

        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        return soup.get_text(separator="\n", strip=True)

    return ""


def split_text(text, size=500, overlap=80):

    chunks = []
    start = 0

    while start < len(text):

        chunk = text[start:start + size]

        if chunk.strip():
            chunks.append(chunk)

        start += size - overlap

    return chunks


def ingest(file):

    text = extract_text(file)
    chunks = split_text(text)

    if not chunks:
        return 0

    embeddings = ollama.embed(
        model="all-minilm",
        input=chunks
    )["embeddings"]

    collection.delete(
        where={"source": file.name}
    )

    collection.add(
        ids=[str(uuid.uuid4()) for _ in chunks],
        documents=chunks,
        embeddings=embeddings,
        metadatas=[
            {"source": file.name, "chunk": i}
            for i in range(len(chunks))
        ]
    )

    return len(chunks)


def generate_answer(question):

    query_embedding = ollama.embed(
        model="all-minilm",
        input=question
    )["embeddings"][0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(3, collection.count())
    )

    context = "\n\n".join(results["documents"][0])

    prompt = f"""
    Answer using only the provided context.
    If information is unavailable, say so.

    Context:
    {context}

    Question:
    {question}
    """

    response = ollama.chat(
        model="tinyllama",
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={
            "num_ctx": 1024,
            "num_predict": 200
        }
    )

    sources = list({
        item["source"]
        for item in results["metadatas"][0]
    })

    return response["message"]["content"], sources


with st.sidebar:

    st.header("Upload Documents")

    uploaded_file = st.file_uploader(
        "Choose PDF or HTML",
        type=["pdf", "html", "htm"]
    )

    if uploaded_file and st.button("Process Document"):

        try:
            with st.spinner("Processing document..."):

                count = ingest(uploaded_file)

            st.success(f"Indexed {count} chunks!")

        except Exception as e:
            st.error(str(e))

    st.write("Total stored chunks:", collection.count())


if "messages" not in st.session_state:
    st.session_state.messages = []


for message in st.session_state.messages:

    with st.chat_message(message["role"]):
        st.write(message["content"])


question = st.chat_input("Ask something about your document...")

if question:

    st.session_state.messages.append({
        "role": "user",
        "content": question
    })

    with st.chat_message("user"):
        st.write(question)

    if collection.count() == 0:

        st.warning("Please upload a document first.")

    else:

        with st.chat_message("assistant"):

            try:
                with st.spinner("Searching and generating..."):

                    answer, sources = generate_answer(question)

                st.write(answer)

                st.write("Sources:", ", ".join(sources))

                st.session_state.messages.append({
                    "role": "assistant",
                    "content": answer + "\nSources: " + ", ".join(sources)
                })

            except Exception as e:
                st.error(str(e))

In [ ]:
!pip install -q pyngrok

In [ ]:
from pyngrok import ngrok

# Configure your authentication token first

In [ ]:
ngrok.set_auth_token("3MdCuvKTMLS7MNjXN3emEuYKl9d_3xW4CwQJyxAdkhefpmhUR")

In [ ]:
!rm -f /root/.config/ngrok/ngrok.yml

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3MdCuvKTMLS7MNjXN3emEuYKl9d_3xW4CwQJyxAdkhefpmhUR")
print("New token configured!")

In [ ]:
public_url = ngrok.connect(8501)

print("Your RAG chatbot URL:", public_url)

In [ ]:
!streamlit run /content/app.py --server.port 8501

In [ ]:
import subprocess
import time

streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Streamlit running:", streamlit_process.poll() is None)

In [ ]:
!cat /content/streamlit.log

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)

print("Your RAG chatbot:", public_url)

In [ ]:
print("Streamlit running:", streamlit_process.poll() is None)

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)

print("Your RAG Chatbot URL:", public_url)

In [ ]:
!curl http://localhost:11434/api/tags

In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(10)

print("Ollama process:", ollama_process.poll())

In [ ]:
!curl http://localhost:11434/api/tags

In [ ]:
import ollama

ollama_client = ollama.Client(
    host="http://127.0.0.1:11434"
)

response = client.chat(
    model="tinyllama",
    messages=[
        {"role": "user", "content": "Say hello"}
    ]
)

print(response["message"]["content"])

In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:11434/api/chat",
    json={
        "model": "tinyllama",
        "messages": [
            {
                "role": "user",
                "content": "Reply with exactly: Hello! I am your AI assistant."
            }
        ],
        "stream": False,
        "options": {
            "temperature": 0,
            "num_predict": 40,
            "num_ctx": 1024
        }
    },
    timeout=180
)

print(response.json()["message"]["content"])

In [ ]:
options={
    "temperature": 0.1,
    "num_ctx": 1024,
    "num_predict": 200
}

In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:11434/api/chat",
    json={
        "model": "tinyllama",
        "messages": [
            {
                "role": "user",
                "content": "Reply with exactly: Hello! I am your AI assistant."
            }
        ],
        "stream": False,
        "options": {
            "temperature": 0,
            "num_predict": 40
        }
    },
    timeout=180
)

print(response.json()["message"]["content"])